In [2]:
pip install PyMuPDF nltk tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
#!/usr/bin/env python3
"""
Batch ABSA pipeline for HBL quarterly reports

Place all PDF reports into a folder named: StockReport
This script will:
 - iterate over every PDF in StockReport/
 - extract text from each PDF using PyMuPDF (fitz)
 - split into sentences (nltk)
 - filter sentences by a list of financial keywords
 - compute a WASI (Weighted Average Sentiment Intensity) score using NLTK VADER
 - write results to /mnt/data/hbl_quarterly_sentiment_scores.csv

Notes:
 - Install requirements if needed:
     pip install PyMuPDF nltk
 - The script will attempt to download required NLTK data (punkt, vader_lexicon).
 - Adjust `financial_sections_keywords` if you want different filtering.

"""

import os
import fitz  # PyMuPDF
import csv
import re
import math
from pathlib import Path

# try importing nltk and its sentiment analyzer
try:
    import nltk
    from nltk.sentiment.vader import SentimentIntensityAnalyzer
except Exception:
    raise ImportError("Please install nltk and its VADER module: pip install nltk")

# --- CONFIG ---
INPUT_FOLDER = "StockReport"  # folder name provided
OUTPUT_CSV = "hbl_quarterly_sentiment_scores.csv"

# Keywords used to focus on financial / narrative sentences (from your notebook)
# Replace old flat keyword list with full aspect‑keyword map
aspects_keywords = {
    "profit": [
        "profit", "earning", "earnings", "income", "net interest income",
        "profit after tax", "eps", "revenue", "topline"
    ],
    "deposits": [
        "deposit", "deposits", "casa", "current accounts",
        "savings accounts", "low-cost deposits"
    ],
    "advances": [
        "advance", "advances", "lending", "loan", "loans", "asset growth"
    ],
    "capital": [
        "capital", "capital adequacy", "car", "tier 1", "ceti",
        "buffers", "capital ratio", "reserves"
    ],
    "npl": [
        "non-performing", "npl", "non performing loans",
        "infection ratio", "non-performing portfolio"
    ],
    "provisions": [
        "provision", "provisioning", "credit loss allowance",
        "impairment", "coverage"
    ],
    "macroeconomic": [
        "inflation", "gdp", "growth", "commodity prices",
        "macroeconomic", "trade deficit", "remittances",
        "policy rate", "sbp", "monetary policy", "imf"
    ],
    "rating": [
        "credit rating", "moodys", "fitch", "outlook revised",
        "stable outlook", "rating agency"
    ],
    "outlook": [
        "outlook", "future", "expected", "guidance", "forecast",
        "trajectory", "prospects"
    ],
    "risk": [
        "risk", "liquidity risk", "market risk", "credit risk",
        "operational risk", "stress", "challenge", "volatility"
    ],
    "digital": [
        "digital", "mobile banking", "internet banking", "pos",
        "cards", "digitalization", "technology"
    ],
    "costs": [
        "cost", "expense", "operating expenses", "cost-to-income",
        "administrative expenses"
    ]
}

# Flatten all keyword lists for filtering
financial_sections_keywords = list({kw for lst in aspects_keywords.values() for kw in lst})

# --------------------------------------------------
# Utility functions
# --------------------------------------------------

def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract text from PDF using PyMuPDF (fitz)."""
    doc = fitz.open(pdf_path)
    text = []
    for page in doc:
        page_text = page.get_text()
        if page_text:
            text.append(page_text)
    return "\n".join(text)


def safe_sent_tokenize(text: str):
    """Sentence tokenize using nltk punkt. Falls back to simple regex split if punkt unavailable."""
    try:
        nltk.data.find('tokenizers/punkt')
    except LookupError:
        nltk.download('punkt')
    try:
        return nltk.tokenize.sent_tokenize(text)
    except Exception:
        # fallback: split on newline and punctuation
        pieces = re.split(r'[\n\r]+', text)
        sentences = []
        for p in pieces:
            s = re.split(r'(?<=[.!?])\s+', p.strip())
            sentences.extend([x for x in s if x])
        return sentences


def compute_wasi_for_sentences(sentences, sia, keywords):
    """
    Compute WASI (Weighted Average Sentiment Intensity).
    - For each sentence, compute VADER compound score (-1..1)
    - Weight by 1 + number_of_keyword_hits in the sentence
    - Return weighted average in range [-1, 1]
    If no sentences provided, return None.
    """
    if not sentences:
        return None

    total_weight = 0.0
    weighted_sum = 0.0

    for s in sentences:
        s_low = s.lower()
        # keyword count (each occurrence counts once)
        keyword_hits = 0
        for kw in keywords:
            if kw in s_low:
                keyword_hits += 1
        weight = 1.0 + keyword_hits  # base weight 1, +1 per matched keyword

        score = sia.polarity_scores(s)['compound']  # -1 .. 1
        weighted_sum += score * weight
        total_weight += weight

    if total_weight == 0:
        return 0.0
    wasi = weighted_sum / total_weight
    return wasi


# --------------------------------------------------
# ASPECT-LEVEL SCORING FUNCTIONS (NEW)
# --------------------------------------------------

def score_aspects(sentences, aspects_keywords, sia):
    """
    Returns a dict:
        { aspect: average_sentiment_score }
    Based on VADER compound score weighted by keyword match count.
    """
    aspect_scores = {a: [] for a in aspects_keywords}

    for s in sentences:
        s_low = s.lower()
        vader = sia.polarity_scores(s)["compound"]

        for aspect, kws in aspects_keywords.items():
            hits = sum(1 for kw in kws if kw in s_low)
            if hits > 0:
                aspect_scores[aspect].append(vader * hits)

    avg_scores = {a: (sum(vals)/len(vals) if vals else 0.0) for a, vals in aspect_scores.items()}
    return avg_scores


def calculate_wasi(aspect_scores):
    """
    WASI = mean of all aspect scores (simple aggregate).
    """
    if not aspect_scores:
        return 0.0
    return sum(aspect_scores.values()) / len(aspect_scores)

# --------------------------------------------------
# Main processing loop
# --------------------------------------------------

def process_all_reports(input_folder: str, output_csv: str):
    input_path = Path(input_folder)
    if not input_path.exists() or not input_path.is_dir():
        raise FileNotFoundError(f"Input folder not found: {input_folder}")

    # prepare NLTK resources
    try:
        nltk.data.find('sentiment/vader_lexicon.zip')
    except LookupError:
        nltk.download('vader_lexicon')

    try:
        nltk.data.find('tokenizers/punkt')
    except LookupError:
        nltk.download('punkt')

    sia = SentimentIntensityAnalyzer()

    results = []  # tuples of (report_name, wasi_score, num_filtered_sentences)

    pdf_files = sorted([p for p in input_path.iterdir() if p.suffix.lower() in ['.pdf']])
    if not pdf_files:
        print(f"No PDF files found in {input_folder}")
        return

    for pdf in pdf_files:
        print(f"Processing: {pdf.name}")
        try:
            raw_text = extract_text_from_pdf(str(pdf))
        except Exception as e:
            print(f"  ERROR extracting {pdf.name}: {e}")
            continue

        sentences = safe_sent_tokenize(raw_text)

        # Filter sentences containing any financial keyword
        filtered_sentences = [s for s in sentences if any(kw in s.lower() for kw in financial_sections_keywords)]

        if not filtered_sentences:
            # fallback: use all sentences but prefer those longer than 40 characters
            candidate_sentences = [s for s in sentences if len(s.strip()) > 40]
            filtered_sentences = candidate_sentences[:200]  # limit to 200 for speed

        wasi_score = compute_wasi_for_sentences(filtered_sentences, sia, financial_sections_keywords)
        if wasi_score is None:
            wasi_score = 0.0

        results.append((pdf.name, wasi_score, len(filtered_sentences)))
        print(f"  Filtered sentences: {len(filtered_sentences)}  WASI: {wasi_score:.4f}")

    # Write CSV
    print(f"Writing results to: {output_csv}")
    outdir = Path(output_csv).parent
    outdir.mkdir(parents=True, exist_ok=True)

    with open(output_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['report_name', 'wasi_score', 'num_filtered_sentences'])
        for r in results:
            writer.writerow([r[0], f"{r[1]:.6f}", r[2]])

    print("Done.")
    return output_csv


if __name__ == '__main__':
    print("Starting batch ABSA pipeline...")
    try:
        csv_path = process_all_reports(INPUT_FOLDER, OUTPUT_CSV)
        print(f"Results saved to: {csv_path}")
    except Exception as e:
        print(f"Fatal error: {e}")


Starting batch ABSA pipeline...
Processing: First_Quarter-Financial_statements_-_Group_2015.pdf
  Filtered sentences: 230  WASI: 0.1523
Processing: First_Quarter-Financial_statements_-_Holding_2015.pdf
  Filtered sentences: 230  WASI: 0.1449
Processing: HBL Quarterly Report March 31, 2025.pdf
  Filtered sentences: 1089  WASI: 0.1388
Processing: HBL_First_Quarter_-_Financial_Statements.pdf
  Filtered sentences: 488  WASI: 0.1053
Processing: HBL_HALF_YEARLY_JUN_2017.pdf
  Filtered sentences: 552  WASI: 0.1138
Processing: HBL_Half_Yearly_June_2016.pdf
  Filtered sentences: 535  WASI: 0.1164
Processing: HBL_Half_Yearly_Report_-_June_30,_2021.pdf
  Filtered sentences: 976  WASI: 0.1287
Processing: HBL_Half_Yearly_Report_-_June_30,_2022.pdf
  Filtered sentences: 1001  WASI: 0.1258
Processing: HBL_Half_Yearly_Report_-_June_30,_2023.pdf
  Filtered sentences: 1036  WASI: 0.1256
Processing: HBL_Half_Yearly_Report_-_June_30,_2024.pdf
  Filtered sentences: 1248  WASI: 0.1518
Processing: HBL_Half_Y